# EXOHUNTER: Exoplanet Candidate Ranking

NASA's TESS telescope finds thousands of "maybe planets". We teach a computer to rank them by how planet-like they look, explain why, check our own work for bias, and look at real telescope data for the top picks.

**Sections**
0. Setup  1. Download data  2. Labels and train/test split  3. Baseline  4. XGBoost model
5. Fair comparison  6. Explain with SHAP and rank  7. Bias tests  8. (reserved)
9. Light-curve vetting of top candidates  10. BLS timing re-check  11. Light-curve feature pilot  12. Which clue helps?

## Section 0: Setup (run this FIRST every time)

In [ ]:
# shap = explains model scores, lightkurve = downloads TESS brightness data.
!pip install -q shap lightkurve

import os
import numpy as np
import pandas as pd
from google.colab import drive      # lets Colab talk to Google Drive

# Connect Google Drive (a popup asks permission; click Allow).
drive.mount('/content/drive')

# ALL project files live here. Drive is permanent, Colab's own disk is not.
PROJECT = '/content/drive/MyDrive/exohunter'
for sub in ['data', 'models', 'plots', 'vetting', 'lcfeatures']:
    os.makedirs(f'{PROJECT}/{sub}', exist_ok=True)   # exist_ok = no complaint if it exists

# ---- The "clues" (features) the model may look at ----
# Transit-only clues: what a telescope light curve gives directly.
transit_only = ['pl_orbper',     # period: days between dips
                'pl_trandurh',   # how many hours each dip lasts
                'pl_trandep',    # how deep the dip is (parts per million)
                'st_tmag']       # star brightness (smaller number = brighter)
# Star clues: facts about the star itself.
star_cols = ['st_teff', 'st_logg', 'st_rad', 'st_dist']
transit_star = transit_only + star_cols
# Calculated planet columns (NASA works these out from the others).
calc_cols = ['pl_rade', 'pl_insol', 'pl_eqt']
all_clues = transit_star + calc_cols
features = all_clues                       # the 11 clues used in the first model

def make_xgb():
    # The same model settings are used everywhere so comparisons are fair.
    import xgboost as xgb
    return xgb.XGBClassifier(
        n_estimators=300,       # 300 small decision trees
        learning_rate=0.05,     # learn slowly and carefully
        max_depth=4,            # each tree asks at most 4 questions in a row
        subsample=0.8,          # each tree sees a random 80% of rows
        colsample_bytree=0.8,   # ... and a random 80% of clues
        eval_metric='logloss',
        random_state=42)        # same results every run

print('Setup done. Project folder:', PROJECT)

## Section 1: Download NASA's candidate table

In [ ]:
# A "DataFrame" is just a table (pandas = Excel for Python).
RAW_PATH = f'{PROJECT}/data/toi_raw.csv'
URL = ('https://exoplanetarchive.ipac.caltech.edu/TAP/sync'
       '?query=select+*+from+toi&format=csv')

if os.path.exists(RAW_PATH):                 # already downloaded? then just load it
    raw = pd.read_csv(RAW_PATH)
    print('Loaded saved file from Drive.')
else:
    raw = pd.read_csv(URL)                   # download straight into a table
    raw.to_csv(RAW_PATH, index=False)        # keep a copy in Drive
    print('Downloaded and saved to Drive.')

print('Rows (candidates) and columns (facts):', raw.shape)
# tfopwg_disp is NASA's verdict. CP/KP = real, FP/FA = fake, PC/APC = undecided.
print(raw['tfopwg_disp'].value_counts(dropna=False))

## Section 2: Labels and the train/test split

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

raw = pd.read_csv(RAW_PATH)

# Turn NASA's verdict codes into labels: real = 1, fake = 0.
label_map = {'CP': 1, 'KP': 1, 'FP': 0, 'FA': 0}
raw['label'] = raw['tfopwg_disp'].map(label_map)    # PC/APC become NaN (= unknown)

labeled = raw[raw['label'].notna()].copy()                 # we KNOW the answer: teach with these
to_rank = raw[raw['tfopwg_disp'].isin(['PC', 'APC'])].copy()  # undecided: rank these later

# Columns we keep. pl_tranmid = time of one dip, pl_trandeperr1 = depth uncertainty.
keep_cols = ['tid', 'toi', 'tfopwg_disp', 'label', 'pl_tranmid', 'pl_trandeperr1'] + features
labeled = labeled[keep_cols]
to_rank = to_rank[keep_cols]

# 80% study set / 20% exam set. All candidates from the SAME star stay together
# (groups=tid) so the exam can't be "leaked" by a sibling the model studied.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(labeled, groups=labeled['tid']))
train = labeled.iloc[train_idx].copy()
test = labeled.iloc[test_idx].copy()

train.to_csv(f'{PROJECT}/data/train.csv', index=False)
test.to_csv(f'{PROJECT}/data/test.csv', index=False)
to_rank.to_csv(f'{PROJECT}/data/to_rank.csv', index=False)

print('Labeled:', len(labeled), '| undecided to rank:', len(to_rank))
print('Study set:', len(train), '| exam set:', len(test))
print('Real vs fake in labeled data:\n', labeled['label'].value_counts())
print('\nPercent of empty cells per clue (labeled data):')
print((labeled[features].isna().mean() * 100).round(1).sort_values(ascending=False))

## Section 3: The old-school baseline
Idea: a deeper dip is more likely to be a real planet. If ML can't beat this, ML wasn't worth it.

In [ ]:
from sklearn.metrics import roc_auc_score    # the grade: 0.5 = coin flip, 1.0 = perfect

train = pd.read_csv(f'{PROJECT}/data/train.csv')
test = pd.read_csv(f'{PROJECT}/data/test.csv')

# Baseline A: rank by depth alone.
score_a = test['pl_trandep'].fillna(test['pl_trandep'].min())

# Baseline B: depth divided by its uncertainty (signal-to-noise).
err = test['pl_trandeperr1'].abs().replace(0, np.nan)
score_b = (test['pl_trandep'] / err)
score_b = score_b.fillna(score_b.min())

auc_a = roc_auc_score(test['label'], score_a)
auc_b = roc_auc_score(test['label'], score_b)
print(f'Baseline A (depth only)    AUC = {auc_a:.3f}')
print(f'Baseline B (depth / error) AUC = {auc_b:.3f}')

# The better baseline is "the number ML has to beat".
test['baseline_score'] = score_a if auc_a >= auc_b else score_b
BASELINE_AUC = max(auc_a, auc_b)
print(f'\nNumber to beat: {BASELINE_AUC:.3f}')

## Section 4: Train the ML model (XGBoost)
We show the computer the study set (with answers), then test it on the exam set (without answers).

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, cross_val_score

X_train, y_train = train[features], train['label'].astype(int)
X_test, y_test = test[features], test['label'].astype(int)

model = make_xgb()

# Practice tests inside the study set (5-fold cross-validation).
# GroupKFold keeps candidates from the same star together (no cheating).
cv_scores = cross_val_score(model, X_train, y_train, groups=train['tid'],
                            cv=GroupKFold(n_splits=5), scoring='roc_auc')
print('Practice-test AUCs:', cv_scores.round(3), '| average', round(cv_scores.mean(), 3))

model.fit(X_train, y_train)                                  # the actual learning
test['ml_score'] = model.predict_proba(X_test)[:, 1]         # "chance it's real"
print(f"\nFINAL EXAM: baseline {BASELINE_AUC:.3f}  vs  ML {roc_auc_score(y_test, test['ml_score']):.3f}")
model.save_model(f'{PROJECT}/models/xgb_all_clues.json')

# Which clues did the model use most?
imp = pd.Series(model.feature_importances_, index=features).sort_values()
imp.plot(kind='barh', figsize=(7, 5), title='Which clues did the model use most?')
plt.tight_layout()
plt.savefig(f'{PROJECT}/plots/feature_importance.png', dpi=150)
plt.show()

## Section 5: Fair comparison
A tougher baseline (logistic regression) and three clue groups, so we can see where the score really comes from.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve

groups = {'Transit only': transit_only,
          'Transit + star': transit_star,
          'All clues': all_clues}

# These columns span huge ranges, so the simple model sees their log10.
log_cols = ['pl_orbper', 'pl_trandurh', 'pl_trandep', 'pl_rade', 'pl_insol', 'st_rad', 'st_dist']
def prepare(df, cols):
    out = df[cols].copy()
    for c in cols:
        if c in log_cols:
            out[c] = np.log10(out[c].where(out[c] > 0))   # can't log zero/negatives -> empty
    return out

results = []
for name, cols in groups.items():
    # Simple model: fill empty cells with the median, scale, then logistic regression.
    lr = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                       LogisticRegression(max_iter=1000))
    lr.fit(prepare(train, cols), y_train)
    lr_scores = lr.predict_proba(prepare(test, cols))[:, 1]

    xg = make_xgb()
    xg.fit(train[cols], y_train)
    xg_scores = xg.predict_proba(test[cols])[:, 1]

    key = name.replace(' ', '_').replace('+', 'and')
    test[f'lr_{key}'], test[f'xgb_{key}'] = lr_scores, xg_scores
    results.append({'Clue group': name,
                    'Logistic AUC': round(roc_auc_score(y_test, lr_scores), 3),
                    'XGBoost AUC': round(roc_auc_score(y_test, xg_scores), 3)})

res_df = pd.DataFrame(results)
print(f'Old-school baseline AUC: {BASELINE_AUC:.3f}')
display(res_df)
res_df.to_csv(f'{PROJECT}/data/model_comparison.csv', index=False)

# ROC curves: the closer to the top-left corner, the better the ranking.
plt.figure(figsize=(7, 6))
for label, col in [('Old-school baseline', 'baseline_score'),
                   ('Logistic (all clues)', 'lr_All_clues'),
                   ('XGBoost (transit only)', 'xgb_Transit_only'),
                   ('XGBoost (all clues)', 'xgb_All_clues')]:
    fpr, tpr, _ = roc_curve(y_test, test[col])
    plt.plot(fpr, tpr, label=f'{label} (AUC {roc_auc_score(y_test, test[col]):.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Coin flip')
plt.xlabel('False alarm rate'); plt.ylabel('Hit rate'); plt.legend(loc='lower right')
plt.title('ROC curves on the exam set'); plt.tight_layout()
plt.savefig(f'{PROJECT}/plots/roc_curves.png', dpi=150); plt.show()

## Section 6: Explain the scores (SHAP) and rank the undecided candidates
We use the **transit + star** model: it skips the three calculated planet columns, so explanations aren't circular.

In [ ]:
import shap

to_rank = pd.read_csv(f'{PROJECT}/data/to_rank.csv')

final_model = make_xgb()
final_model.fit(train[transit_star], y_train)
print('Final model exam AUC:',
      round(roc_auc_score(y_test, final_model.predict_proba(test[transit_star])[:, 1]), 3))
final_model.save_model(f'{PROJECT}/models/final_model.json')

# SHAP: for every candidate, how much did each clue push the score up or down?
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(test[transit_star])
shap.summary_plot(shap_values, test[transit_star], show=False)   # red = high clue value
plt.tight_layout()
plt.savefig(f'{PROJECT}/plots/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# Score and rank the undecided candidates (1 = most promising).
to_rank['score'] = final_model.predict_proba(to_rank[transit_star])[:, 1]
to_rank['rank'] = to_rank['score'].rank(ascending=False, method='first')
ranked = to_rank.sort_values('rank').reset_index(drop=True)
ranked.to_csv(f'{PROJECT}/data/ranked_candidates.csv', index=False)
display(ranked[['rank', 'toi', 'tfopwg_disp', 'score'] + transit_star].head(15))

# Why did candidate #1 get its score?
top = ranked.loc[[0], transit_star]
why = pd.DataFrame({'clue': transit_star, 'value': top.iloc[0].values,
                    'push_on_score': explainer.shap_values(top)[0].round(3)})
display(why.sort_values('push_on_score', key=abs, ascending=False))
print('NOTE: scores order candidates. They are NOT probabilities of being a real planet.')

## Section 7: Bias tests
Nearby stars get followed up (and confirmed) more often, so a model can learn "close star = real planet". We measure that and build two alternative rankings:
* **Transit-only model** (cannot see star distance/temperature/size)
* **Distance-matched model** (trained on a set where real and fake are balanced in every distance group)

In [ ]:
from scipy.stats import spearmanr

ranked = pd.read_csv(f'{PROJECT}/data/ranked_candidates.csv')

# ---- 10 distance groups with equal numbers of stars ----
edges = np.quantile(train['st_dist'].dropna(), np.linspace(0, 1, 11))
edges[0], edges[-1] = -np.inf, np.inf
edges = np.unique(edges)

# PROOF of the bias: what fraction of each distance group is real?
tmp = train.dropna(subset=['st_dist']).copy()
tmp['dist_bin'] = pd.cut(tmp['st_dist'], bins=edges, labels=False)
print('Fraction REAL per distance group (0.5 would mean no bias):')
display(tmp.groupby('dist_bin').agg(median_pc=('st_dist', 'median'),
                                    fraction_real=('label', 'mean'),
                                    n=('label', 'size')).round(2))

def match(df, seed=42):
    # In every distance group keep equal numbers of real and fake candidates.
    df = df.dropna(subset=['st_dist']).copy()
    df['dist_bin'] = pd.cut(df['st_dist'], bins=edges, labels=False)
    parts = []
    for _, g in df.groupby('dist_bin'):
        real, fake = g[g['label'] == 1], g[g['label'] == 0]
        n = min(len(real), len(fake))
        if n:
            parts += [real.sample(n, random_state=seed), fake.sample(n, random_state=seed)]
    return pd.concat(parts)

train_m, test_m = match(train), match(test)
matched_model = make_xgb()
matched_model.fit(train_m[transit_star], train_m['label'].astype(int))
transit_model = make_xgb()
transit_model.fit(train[transit_only], y_train)
matched_model.save_model(f'{PROJECT}/models/matched_model.json')
transit_model.save_model(f'{PROJECT}/models/transit_model.json')

# ---- Grade the star-aware and matched models on BOTH exams ----
def auc(m, df):
    return roc_auc_score(df['label'].astype(int), m.predict_proba(df[transit_star])[:, 1])
print('\n                 Full exam   Distance-balanced exam')
print(f'Star-aware:       {auc(final_model, test):.3f}       {auc(final_model, test_m):.3f}')
print(f'Distance-matched: {auc(matched_model, test):.3f}       {auc(matched_model, test_m):.3f}')

# ---- Three rankings of the undecided candidates ----
ranked['score_transit'] = transit_model.predict_proba(ranked[transit_only])[:, 1]
ranked['score_matched'] = matched_model.predict_proba(ranked[transit_star])[:, 1]
ranked['rank_transit'] = ranked['score_transit'].rank(ascending=False, method='first')
ranked['rank_matched'] = ranked['score_matched'].rank(ascending=False, method='first')

print('\nWhat stars are in each top-100?')
for name, col in [('Star-aware', 'rank'), ('Transit-only', 'rank_transit'), ('Distance-matched', 'rank_matched')]:
    t100 = ranked.nsmallest(100, col)
    print(f'{name:17s} median distance {t100["st_dist"].median():5.0f} pc | median temp {t100["st_teff"].median():5.0f} K')
print(f'{"All undecided":17s} median distance {ranked["st_dist"].median():5.0f} pc | median temp {ranked["st_teff"].median():5.0f} K')

corr, _ = spearmanr(ranked['rank'], ranked['rank_transit'])
print(f'\nRanking similarity (star-aware vs transit-only): {corr:.2f}')

# Candidates in BOTH the star-aware and transit-only top 100 = the most robust picks.
a100, b100 = ranked.nsmallest(100, 'rank'), ranked.nsmallest(100, 'rank_transit')
consensus = a100[a100['toi'].isin(b100['toi'])]
print('Consensus candidates:', len(consensus))
consensus.to_csv(f'{PROJECT}/data/consensus_candidates.csv', index=False)
ranked.to_csv(f'{PROJECT}/data/ranked_candidates_all3.csv', index=False)

# ---- Frozen, dated snapshot so you can check our predictions against NASA later ----
from datetime import date
snap = ranked[['toi', 'tid', 'tfopwg_disp', 'rank', 'rank_transit', 'rank_matched',
               'score', 'score_transit', 'score_matched']].copy()
snap['snapshot_date'] = str(date.today())
snap.to_csv(f'{PROJECT}/data/prediction_snapshot_{date.today()}.csv', index=False)
print('Snapshot saved. In a few months, re-download the NASA table and see which candidates were decided.')

## Section 9: Look at real TESS light curves for the top candidates
For each candidate we download the star's brightness over time, remove slow drift, **fold** it on the catalog schedule (stack the repeating dips), and run three checks:
1. **Signal strength (SNR)**
2. **Odd vs even dips** (different depths can mean two stars eclipsing)
3. **Secondary dip** (a second dip halfway between main dips is a binary-star warning)

A flag only counts if it is both **significant (over 3 sigma)** and **big (over 20% of the dip depth)**, and every test needs at least 50 data points. This is a sanity check, **not** proof of a planet.

In [ ]:
import lightkurve as lk
import time

MIN_POINTS = 50   # a test must have at least this many data points to count

def depth_and_err(f, mask, noise):
    # Depth (ppm) of the points in 'mask' and its uncertainty. NaN if too few points.
    n = int(mask.sum())
    if n < MIN_POINTS:
        return np.nan, np.nan, n
    depth = (1 - f[mask].mean()) * 1e6        # how far below 1.0, in parts per million
    err = noise / np.sqrt(n) * 1e6            # more points = smaller uncertainty
    return depth, err, n

def run_checks(t, f, P, t0, dur):
    # Fold: phase = distance (days) from the nearest expected dip.
    phase = ((t - t0 + 0.5 * P) % P) - 0.5 * P
    epoch = np.round((t - t0) / P)                 # which dip number is each point near
    in_tr = np.abs(phase) < dur / 4                # middle of the dip
    far = np.abs(phase) > dur                      # clearly away from the dip
    if far.sum() < MIN_POINTS:
        return None
    noise = np.std(f[far])                         # the star's normal wobble
    depth, err, n_in = depth_and_err(f, in_tr, noise)
    out = dict(depth=depth, err=err, snr=np.nan, odd_even_sigma=np.nan,
               odd_even_pct=np.nan, secondary_sigma=np.nan, secondary_pct=np.nan)
    if np.isnan(depth):
        return out
    out['snr'] = depth / err
    # Odd vs even dips
    d_odd, e_odd, _ = depth_and_err(f, in_tr & (epoch % 2 == 1), noise)
    d_even, e_even, _ = depth_and_err(f, in_tr & (epoch % 2 == 0), noise)
    if not (np.isnan(d_odd) or np.isnan(d_even)):
        out['odd_even_sigma'] = abs(d_odd - d_even) / np.sqrt(e_odd**2 + e_even**2)
        if depth > 0:
            out['odd_even_pct'] = 100 * abs(d_odd - d_even) / depth
    # Secondary dip halfway between main dips
    sec = np.abs(np.abs(phase) - P / 2) < dur / 4
    d_sec, e_sec, _ = depth_and_err(f, sec, noise)
    if not np.isnan(d_sec):
        out['secondary_sigma'] = d_sec / e_sec
        if depth > 0:
            out['secondary_pct'] = 100 * d_sec / depth
    return out

def get_lightcurve(tic, P, t0, dur, n_sectors, cache=None):
    # Return (time, flux, sectors_used), or None if the star has no 2-minute data.
    if cache and os.path.exists(cache):                  # saved earlier? reuse it
        s = pd.read_csv(cache)
        return s['time'].values, s['flux'].values, int(s['n_sectors'].iloc[0])
    found = lk.search_lightcurve(f'TIC {int(tic)}', mission='TESS', author='SPOC', exptime=120)
    if len(found) == 0:
        return None
    n = min(len(found), n_sectors)
    lc = found[:n].download_all().stitch().remove_nans()
    # Flatten = remove slow drift, but hide the dips from the flattening so they survive.
    in_dip = lc.create_transit_mask(period=P, transit_time=t0, duration=dur * 1.5)
    flat = lc.flatten(window_length=721, mask=in_dip)
    flat = flat.remove_outliers(sigma_upper=4, sigma_lower=20)
    t = np.asarray(flat.time.value, dtype=float)
    f = np.asarray(flat.flux.value, dtype=float)
    if cache:
        pd.DataFrame({'time': t, 'flux': f, 'n_sectors': n}).to_csv(cache, index=False)
    return t, f, n

def get_verdict(r, catalog_depth):
    # Turn the numbers into a cautious plain-English verdict.
    if r is None or np.isnan(r['snr']):
        return 'inconclusive (too little data)'
    expected_snr = catalog_depth / r['err']            # how clear it SHOULD be
    if r['snr'] < 7:
        if expected_snr >= 7:
            return 'dip NOT found at catalog timing (run BLS re-check)'
        return 'inconclusive (too shallow to test)'
    if np.isnan(r['odd_even_sigma']) or np.isnan(r['secondary_sigma']):
        return 'dip present, binary checks not possible'
    big_odd = r['odd_even_sigma'] > 3 and r['odd_even_pct'] > 20
    big_sec = r['secondary_sigma'] > 3 and r['secondary_pct'] > 20
    if big_odd or big_sec:
        return 'LIKELY eclipsing binary (large difference)'
    if r['odd_even_sigma'] > 3 or r['secondary_sigma'] > 3:
        return 'small flag only (re-check with more data)'
    if r['snr'] < 10:
        return 'passes, but marginal SNR (7-10)'
    return 'passes simple checks (clear)'

def save_plot(t, f, P, t0, dur, name, r):
    # Left: zoom on the dip. Right: whole orbit (a second dip at the edges is a warning).
    phase = ((t - t0 + 0.5 * P) % P) - 0.5 * P
    def binned(x, y, edges):
        idx = np.digitize(x, edges)
        c, m = [], []
        for i in range(1, len(edges)):
            if (idx == i).sum() > 2:
                c.append((edges[i - 1] + edges[i]) / 2); m.append(np.median(y[idx == i]))
        return np.array(c), np.array(m)
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    half = min(P / 2, 4 * dur)
    z = np.abs(phase) < half
    ax[0].scatter(phase[z] * 24, f[z], s=2, color='lightgray')
    cx, cy = binned(phase[z], f[z], np.linspace(-half, half, 41))
    ax[0].plot(cx * 24, cy, 'r-o', ms=3)
    d = r['depth'] if (r and not np.isnan(r['depth'])) else 500
    d = max(d, 500)
    ax[0].set_ylim(1 - 4 * d * 1e-6, 1 + 2 * d * 1e-6)
    ax[0].set_title(f'{name}: zoom on the dip'); ax[0].set_xlabel('Hours from expected dip')
    cx2, cy2 = binned(phase, f, np.linspace(-P / 2, P / 2, 101))
    ax[1].plot(cx2 / P, cy2, 'b-o', ms=3)
    ax[1].set_title('Whole orbit'); ax[1].set_xlabel('Fraction of orbit (0 = main dip)')
    plt.tight_layout(); plt.savefig(f'{PROJECT}/plots/vet_{name}.png', dpi=110); plt.close(fig)

# ---------- Pick candidates that rank high in ALL THREE rankings ----------
HOW_MANY, MAX_TMAG, MIN_DEPTH = 30, 11.0, 150   # bright stars, reasonably deep dips
ranked = pd.read_csv(f'{PROJECT}/data/ranked_candidates_all3.csv')
ranked['worst_rank'] = ranked[['rank', 'rank_transit', 'rank_matched']].max(axis=1)
ranked = ranked.dropna(subset=['pl_tranmid', 'pl_orbper', 'pl_trandurh', 'pl_trandep', 'st_tmag', 'tid'])
ok = ranked[(ranked['st_tmag'] <= MAX_TMAG) & (ranked['pl_trandep'] >= MIN_DEPTH)]
picks = ok.sort_values('worst_rank').head(HOW_MANY).reset_index(drop=True)
picks.to_csv(f'{PROJECT}/data/vetting_picks.csv', index=False)
print(f'{len(ok)} candidates pass the filters; checking {len(picks)}.')

# ---------- Main loop: one star at a time, SAVING after each (safe from disconnects) ----------
MAX_SECTORS, MAX_MINUTES = 6, 25
start = time.time()
for i, row in picks.iterrows():
    name = f"TOI_{row['toi']}"
    result_file = f'{PROJECT}/vetting/{name}_result.csv'
    if os.path.exists(result_file):                      # finished earlier -> skip
        continue
    if (time.time() - start) / 60 > MAX_MINUTES:
        print('Time budget reached. Press play again to continue.'); break
    P, t0, dur = row['pl_orbper'], row['pl_tranmid'] - 2457000, row['pl_trandurh'] / 24
    try:
        got = get_lightcurve(row['tid'], P, t0, dur, MAX_SECTORS, f'{PROJECT}/vetting/{name}_lc.csv')
        res = {'toi': row['toi'], 'catalog_depth_ppm': round(row['pl_trandep'])}
        if got is None:
            res['verdict'] = 'no 2-minute data'
        else:
            t, f, n = got
            r = run_checks(t, f, P, t0, dur)
            save_plot(t, f, P, t0, dur, name, r)
            if r:
                res.update({'measured_depth_ppm': r['depth'], 'n_sectors': n, 'SNR': r['snr'],
                            'odd_even_sigma': r['odd_even_sigma'], 'odd_even_pct': r['odd_even_pct'],
                            'secondary_sigma': r['secondary_sigma'], 'secondary_pct': r['secondary_pct']})
            res['verdict'] = get_verdict(r, row['pl_trandep'])
        pd.DataFrame([res]).to_csv(result_file, index=False)   # SAVE NOW
        print(f"[{i + 1}/{len(picks)}] {name}: {res['verdict']}")
    except Exception as e:
        print(f'{name} failed ({e}). It will be retried next run.')

In [ ]:
# Combine every per-star result file into one table.
import glob
files = glob.glob(f'{PROJECT}/vetting/*_result.csv')
vet = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
vet = vet.merge(picks[['toi', 'rank', 'rank_transit', 'rank_matched', 'worst_rank']], on='toi', how='left')
vet = vet.sort_values('worst_rank')
vet.to_csv(f'{PROJECT}/data/vetting_table.csv', index=False)
print(vet['verdict'].value_counts())
display(vet.round(1))

## Section 10: BLS timing re-check
Catalog periods can be a few minutes off. Over years of data that smears the dips and can hide a real one (this happened with TOI 3344.01). **BLS** ignores the catalog schedule and searches for the period that makes the dips pile up best.

In [ ]:
from astropy.timeseries import BoxLeastSquares

def bls_recheck(toi):
    row = picks[np.isclose(picks['toi'], toi)].iloc[0]
    P_cat, dur_cat = row['pl_orbper'], row['pl_trandurh'] / 24
    d = pd.read_csv(f'{PROJECT}/vetting/TOI_{toi}_lc.csv')
    t, f = d['time'].values, d['flux'].values
    # Average 2-minute points into 10-minute points (much faster, barely changes a deep dip).
    idx = np.floor((t - t.min()) / (10 / 1440)).astype(int)
    b = pd.DataFrame({'i': idx, 't': t, 'f': f}).groupby('i').agg(t=('t', 'mean'), f=('f', 'mean'), n=('f', 'size'))
    b = b[b['n'] >= 3]
    tb, fb = b['t'].values, b['f'].values
    durations = np.array([1.5, 2, 3, 4, 5]) / 24
    bls = BoxLeastSquares(tb, fb)
    factor = 1.0                        # coarser period list until it is a sane size
    while True:
        periods = bls.autoperiod(durations, minimum_period=P_cat * 0.9, maximum_period=P_cat * 1.1,
                                 frequency_factor=factor)
        if len(periods) < 300000:
            break
        factor *= 2
    res = bls.power(periods, durations, objective='snr')
    k = np.argmax(res.power)
    P_new, t0_new, dur_new = res.period[k], res.transit_time[k], res.duration[k]
    r = run_checks(t, f, P_new, t0_new, dur_new)       # redo the checks on the NEW schedule
    print(f'TOI {toi}: catalog period {P_cat:.5f} d -> BLS {P_new:.5f} d '
          f'({abs(P_new - P_cat) * 1440:.1f} min apart)')
    return r, row['pl_trandep']

# Re-check every star the first pass said "dip NOT found".
for toi in vet.loc[vet['verdict'].str.startswith('dip NOT found'), 'toi']:
    r, cat_depth = bls_recheck(toi)
    print('   new verdict:', get_verdict(r, cat_depth))

## Section 11: Pilot: do light-curve measurements help the model?
For ~300 labeled candidates we measure four light-curve clues from the **first 2 sectors** (the same amount of data for every star, so the model can't learn "has lots of data"):
`lc_snr`, `lc_depth_ratio` (measured depth / catalog depth), `lc_odd_even_pct`, `lc_secondary_pct`.

In [ ]:
PILOT_SIZE, N_SECTORS = 300, 2

lab = pd.concat([pd.read_csv(f'{PROJECT}/data/train.csv'), pd.read_csv(f'{PROJECT}/data/test.csv')],
                ignore_index=True)
lab = lab.dropna(subset=['pl_tranmid', 'pl_orbper', 'pl_trandurh', 'pl_trandep', 'tid'])
eligible = lab[(lab['pl_orbper'] <= 20) & (lab['st_tmag'] <= 12)]     # short periods, bright stars
pilot = eligible.sample(min(PILOT_SIZE, len(eligible)), random_state=42).reset_index(drop=True)
pilot.to_csv(f'{PROJECT}/data/pilot_picks.csv', index=False)
print('Eligible:', len(eligible), '| picked:', len(pilot))

start = time.time()
for i, row in pilot.iterrows():
    result_file = f"{PROJECT}/lcfeatures/TOI_{row['toi']}_result.csv"
    if os.path.exists(result_file):
        continue
    if (time.time() - start) / 60 > 25:
        print('Time budget reached. Press play again to continue.'); break
    P, t0, dur = row['pl_orbper'], row['pl_tranmid'] - 2457000, row['pl_trandurh'] / 24
    try:
        got = get_lightcurve(row['tid'], P, t0, dur, N_SECTORS, cache=None)   # no cache: saves disk
        res = {'toi': row['toi']}
        if got is None:
            res['status'] = 'no 2-minute data'
        else:
            r = run_checks(got[0], got[1], P, t0, dur)
            if r is None:
                res['status'] = 'too little data'
            else:
                res.update({'status': 'ok', 'lc_snr': r['snr'],
                            'lc_depth_ratio': r['depth'] / row['pl_trandep'],
                            'lc_odd_even_pct': r['odd_even_pct'], 'lc_secondary_pct': r['secondary_pct']})
        pd.DataFrame([res]).to_csv(result_file, index=False)    # SAVE NOW
    except Exception as e:
        print(f"TOI {row['toi']} failed ({e}); will retry next run.")

n_done = len(glob.glob(f'{PROJECT}/lcfeatures/*_result.csv'))
print(f'Finished {n_done} of {len(pilot)}.')

## Section 12: Which light-curve clue helps?
Grouped 5x5 cross-validation on the pilot (same star never on both sides). Reshuffles reuse the same rows, so treat gaps under about 0.02 as ties.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
import xgboost as xgb

res = pd.concat([pd.read_csv(f) for f in glob.glob(f'{PROJECT}/lcfeatures/*_result.csv')], ignore_index=True)
every = pilot.merge(res, on='toi', how='left')

# Is MISSING 2-minute data linked to the label? (It is: most missing ones are fakes.)
every['has_2min'] = every['status'] != 'no 2-minute data'
print('Share with 2-minute data, by label:')
print(every.groupby('label')['has_2min'].agg(['mean', 'sum', 'count']).round(2))

d = every[every['status'] == 'ok'].reset_index(drop=True)
y = d['label'].astype(int).values
lc_cols = ['lc_snr', 'lc_depth_ratio', 'lc_odd_even_pct', 'lc_secondary_pct']
print(f'\nUsable rows: {len(d)}')
print('\nMedian of each new clue, real (1) vs fake (0):')
print(d.groupby('label')[lc_cols].median().round(2))

def small_model():
    # Small, gentle model because the pilot is small.
    return xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, subsample=0.8,
                             colsample_bytree=0.8, eval_metric='logloss', random_state=42)

def compare(base_cols, base_name):
    sets = {base_name: base_cols,
            '+ secondary only': base_cols + ['lc_secondary_pct'],
            '+ odd/even only': base_cols + ['lc_odd_even_pct'],
            '+ SNR only': base_cols + ['lc_snr'],
            '+ depth ratio only': base_cols + ['lc_depth_ratio'],
            '+ physics pair (secondary + odd/even)': base_cols + ['lc_secondary_pct', 'lc_odd_even_pct'],
            '+ all four': base_cols + lc_cols}
    per_seed = {n: [] for n in sets}
    for seed in range(5):                                    # 5 different shuffles
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
        fold_scores = {n: [] for n in sets}
        for tr, te in cv.split(d, y, groups=d['tid']):
            for n, cols in sets.items():
                m = small_model().fit(d.loc[tr, cols], y[tr])
                fold_scores[n].append(roc_auc_score(y[te], m.predict_proba(d.loc[te, cols])[:, 1]))
        for n in sets:
            per_seed[n].append(np.mean(fold_scores[n]))
    base = np.array(per_seed[base_name])
    print(f'\n=== Baseline: {base_name} (AUC {base.mean():.3f}) ===')
    for n, s in per_seed.items():
        if n != base_name:
            g = np.array(s) - base
            print(f'{n:40s} AUC {np.mean(s):.3f}  gain {g.mean():+.3f}  (range {g.min():+.3f} to {g.max():+.3f})')

compare(transit_only, 'Transit only')
compare(transit_star, 'Transit + star')

## Summary of what to report
* XGBoost beats the depth baseline (AUC about 0.76 to 0.92 depending on clues; baseline about 0.55).
* The top of the ranking depends on how selection bias is handled; neither ranking is "the truth".
* Light-curve checks on 30 top candidates: no large eclipsing-binary signature among the testable ones, but this is a sanity check on a non-random sample.
* The light-curve feature pilot gave a small gain whose cause (physics vs catalog-timing bias) is not established.
* Scores order candidates; they are **not** probabilities. Re-check the frozen snapshot against NASA's verdicts later.